# Narkomfin Type K — Two-Floor Spatial Intelligence (Grid Sampling)

Analyze L1 and L2 of the Narkomfin Type K apartment as a single connected building.

**Method:** Grid-sample each floor plan → build per-floor graphs → union → stitch with stair edges → run spatial analysis across both floors.

**Stair geometry** is loaded from `TheNarkomfinHouse-Ktype-withStairs.obj` — surfaces that span Y=0 (L1) to Y=3 (L2) are identified automatically and their plan centroids become the stair connection points.

## 1. Imports

In [ ]:
import time
import numpy as np

from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Graph import Graph
from topologicpy.Color import Color

print(Helper.Version())

In [ ]:
renderer = "vscode"

## 2. Configuration

In [ ]:
from pathlib import Path

HERE = Path.cwd()
ASSET_DIR  = HERE.parent / '02_graph_analysis' / 'assets'
OUTPUT_DIR = HERE.parent / '02_graph_analysis' / 'output'

OBJ_L1     = ASSET_DIR / 'TheNarkomfinHouse-01.obj'
OBJ_L2     = ASSET_DIR / 'TheNarkomfinHouse-02.obj'
OBJ_STAIRS = ASSET_DIR / 'TheNarkomfinHouse-Ktype-withStairs.obj'

GRID_SIZE     = 1.0   # grid spacing in plan units (smaller = finer & slower)
FLOOR_HEIGHTS = [0, 3]  # Y values in the OBJ (L1=0, L2=3)
FLOOR_NAMES   = ['L1 (Y=0)', 'L2 (Y=3)']
FLOOR_Z_VIS   = [0, 3]  # Z offset for 3D graph visualization (actual height)

print('OBJ files:')
for p in [OBJ_L1, OBJ_L2, OBJ_STAIRS]:
    print(f'  {p.name} — exists: {p.exists()}')

## 3. Utility functions

In [ ]:
from matplotlib.path import Path as MplPath

def points_inside_faces(face_list, test_pts):
    """Ray-casting point-in-polygon: test which points fall inside any face.
    Works correctly for concave polygons (unlike fan-triangulation)."""
    inside = np.zeros(len(test_pts), bool)
    for f in face_list:
        vs = Topology.Vertices(f)
        poly = np.array([(Vertex.X(v), Vertex.Y(v)) for v in vs])
        path = MplPath(poly)
        inside |= path.contains_points(test_pts)
    return inside

def find_closest_node(node_xy, x, y):
    """Index of the grid node closest to (x, y)."""
    d = (node_xy[:, 0] - x) ** 2 + (node_xy[:, 1] - y) ** 2
    return int(d.argmin())

def rk(u, v):
    return (round(float(u), 3), round(float(v), 3))

def make_cell_face(cx, cy, cz, h):
    """Flat square cell centred at (cx, cy, cz) with half-size h."""
    pts = [Vertex.ByCoordinates(cx - h, cy - h, cz),
           Vertex.ByCoordinates(cx + h, cy - h, cz),
           Vertex.ByCoordinates(cx + h, cy + h, cz),
           Vertex.ByCoordinates(cx - h, cy + h, cz)]
    return Face.ByWire(Wire.ByVertices(pts, close=True))

print('Utilities loaded.')

## 4. Load floor plans and identify stair locations

In [ ]:
# Load L1 and L2 floor faces
floor_faces = {}
for lv, obj_path, name in zip(FLOOR_HEIGHTS, [OBJ_L1, OBJ_L2], FLOOR_NAMES):
    result = Topology.ByOBJPath(str(obj_path))
    faces = []
    if isinstance(result, list):
        for item in result:
            if Topology.IsInstance(item, 'Cluster'):
                cf = Topology.Faces(item)
                if cf: faces.extend(cf)
            elif Topology.IsInstance(item, 'Face'):
                faces.append(item)
    else:
        faces = Topology.Faces(result)
    floor_faces[lv] = faces
    print(f'{name}: {len(faces)} faces')

# Load the combined file to extract stair surfaces
result_stairs = Topology.ByOBJPath(str(OBJ_STAIRS))
all_stair_faces = []
if isinstance(result_stairs, list):
    for item in result_stairs:
        if Topology.IsInstance(item, 'Cluster'):
            cf = Topology.Faces(item)
            if cf: all_stair_faces.extend(cf)
        elif Topology.IsInstance(item, 'Face'):
            all_stair_faces.append(item)
else:
    all_stair_faces = Topology.Faces(result_stairs)
print(f'\nStairs file: {len(all_stair_faces)} total faces')

# Identify stair faces: those whose vertices span multiple Z levels
stair_surfaces = []
for f in all_stair_faces:
    zs = set(round(Vertex.Z(v), 1) for v in Topology.Vertices(f))
    if len(zs) > 1:
        stair_surfaces.append(f)

# Extract stair plan-coordinate centroids
stair_locations = []
for f in stair_surfaces:
    verts = Topology.Vertices(f)
    xs = [Vertex.X(v) for v in verts]
    ys = [Vertex.Y(v) for v in verts]
    stair_locations.append(((min(xs)+max(xs))/2, (min(ys)+max(ys))/2))

print(f'Stair surfaces identified: {len(stair_surfaces)}')
for i, (sx, sy) in enumerate(stair_locations):
    print(f'  Stair {i+1}: X={sx:.2f}, Y={sy:.2f}')

## 5. Show raw floor plans

In [ ]:
for lv, name in zip(FLOOR_HEIGHTS, FLOOR_NAMES):
    cluster = Cluster.ByTopologies(floor_faces[lv])
    print(f'{name}:')
    Topology.Show(cluster,
                  camera=[0,0,6],
                  faceColor=[210,210,250],
                  faceOpacity=1,
                  edgeColor='white',
                  edgeWidth=2,
                  showVertices=False,
                  backgroundColor='black',
                  width=800, height=400,
                  renderer=renderer)

## 6. Grid-sample navigable points on each floor

In [ ]:
# Shared bounding box from both floors (using vertex coordinates directly)
all_xs, all_ys = [], []
for lv in FLOOR_HEIGHTS:
    for f in floor_faces[lv]:
        for v in Topology.Vertices(f):
            all_xs.append(Vertex.X(v))
            all_ys.append(Vertex.Y(v))
UMIN, UMAX = min(all_xs), max(all_xs)
VMIN, VMAX = min(all_ys), max(all_ys)
print(f'Plan bounding box: X[{UMIN:.1f}, {UMAX:.1f}]  Y[{VMIN:.1f}, {VMAX:.1f}]')

# Regular grid
us = np.arange(UMIN, UMAX + GRID_SIZE, GRID_SIZE)
vs = np.arange(VMIN, VMAX + GRID_SIZE, GRID_SIZE)
UU, VV = np.meshgrid(us, vs)
GRID_PTS = np.column_stack([UU.ravel(), VV.ravel()])
print(f'Grid: {len(us)}x{len(vs)} = {len(GRID_PTS)} candidate points')

# Filter to navigable (inside floor faces) per floor using ray-casting
floor_valid = {}
for lv, name in zip(FLOOR_HEIGHTS, FLOOR_NAMES):
    mask = points_inside_faces(floor_faces[lv], GRID_PTS)
    floor_valid[lv] = GRID_PTS[mask]
    print(f'  {name}: {len(floor_valid[lv])} navigable nodes')

## 7. Build per-floor graphs + display cells, then stack

In [ ]:
all_v = []           # topologic vertices (stacked by floor in Z)
all_e = []           # topologic edges
floor_index_map = {} # level -> {(round u, round v): global vertex index}
display_faces = []   # flat square cells for heatmap rendering
cell_lookup = {}     # (floor_idx, (round u, round v)) -> display face
H = GRID_SIZE / 2.0

for fi, lv in enumerate(FLOOR_HEIGHTS):
    valid = floor_valid[lv]
    z_vis = FLOOR_Z_VIS[fi]
    idx = {}
    for (u, v) in valid:
        key = rk(u, v)
        idx[key] = len(all_v)
        all_v.append(Vertex.ByCoordinates(float(u), float(v), float(z_vis)))
        cell = make_cell_face(float(u), float(v), float(z_vis), H)
        display_faces.append(cell)
        cell_lookup[(fi, key)] = cell
    floor_index_map[lv] = idx
    ne = 0
    for (u, v) in valid:
        for du, dv in [(GRID_SIZE, 0), (0, GRID_SIZE)]:
            k = rk(u + du, v + dv)
            if k in idx:
                all_e.append(Edge.ByVertices([all_v[idx[rk(u, v)]], all_v[idx[k]]]))
                ne += 1
    print(f'  {FLOOR_NAMES[fi]}: {len(valid)} nodes, {ne} horizontal edges')
print(f'Subtotal: {len(all_v)} nodes, {len(all_e)} horizontal edges')

## 8. Connect floors through stair edges

In [ ]:
stair_edges_added = 0
l1_valid = floor_valid[FLOOR_HEIGHTS[0]]
l2_valid = floor_valid[FLOOR_HEIGHTS[1]]
l1_idx = floor_index_map[FLOOR_HEIGHTS[0]]
l2_idx = floor_index_map[FLOOR_HEIGHTS[1]]

for sx, sy in stair_locations:
    i1 = find_closest_node(l1_valid, sx, sy)
    i2 = find_closest_node(l2_valid, sx, sy)
    key1 = rk(l1_valid[i1, 0], l1_valid[i1, 1])
    key2 = rk(l2_valid[i2, 0], l2_valid[i2, 1])
    gi1 = l1_idx[key1]
    gi2 = l2_idx[key2]
    all_e.append(Edge.ByVertices([all_v[gi1], all_v[gi2]]))
    stair_edges_added += 1

print(f'Added {stair_edges_added} stair edges from {len(stair_locations)} stair locations')
print(f'Total: {len(all_v)} nodes, {len(all_e)} edges')

## 9. Build the combined building graph

In [ ]:
t0 = time.time()
building_graph = Graph.ByVerticesEdges(all_v, all_e)
gverts = Graph.Vertices(building_graph)
gedges = Graph.Edges(building_graph)
print(f'Building graph: {len(gverts)} vertices, {len(gedges)} edges  ({time.time()-t0:.1f}s)')
print(f'Graph density: {Graph.Density(building_graph):.6f}')

## 10. Show the building graph

In [ ]:
Topology.Show(building_graph,
              camera=[1,1,1],
              vertexSize=2,
              vertexColor='red',
              edgeColor='lightgrey',
              backgroundColor='black',
              width=900, height=600,
              renderer=renderer)

## 11. Spatial Analysis — helper to render heatmaps

In [ ]:
def heatmap_from_values(values, title, colorScale='thermal'):
    """Pair each graph vertex with its display cell, colour by value, render."""
    face_val_pairs = []
    for v, val in zip(gverts, values):
        z = Vertex.Z(v)
        fi = FLOOR_Z_VIS.index(round(z)) if round(z) in FLOOR_Z_VIS else 0
        key = rk(Vertex.X(v), Vertex.Y(v))
        cell = cell_lookup.get((fi, key))
        if cell is not None:
            face_val_pairs.append((cell, val))

    vals = [v for _, v in face_val_pairs]
    mn, mx = float(min(vals)), float(max(vals))
    if mx == mn: mx = mn + 1e-9
    for f, val in face_val_pairs:
        col = Color.AnyToHex(Color.ByValueInRange(float(val), minValue=mn, maxValue=mx, colorScale=colorScale))
        d = Topology.Dictionary(f)
        d = Dictionary.SetValueAtKey(d, 'hm_color', col)
        Topology.SetDictionary(f, d)
    faces = [f for f, _ in face_val_pairs]
    print(f'{title}: {len(faces)} cells, range [{mn:.4f}, {mx:.4f}]')
    Topology.Show(faces,
                  faceColorKey='hm_color',
                  faceOpacity=1,
                  showEdges=False,
                  showVertices=False,
                  camera=[0,0,6],
                  backgroundColor='black',
                  width=900, height=600,
                  renderer=renderer)

print('Heatmap helper ready.')

## 12. Shortest Path (cross-floor)

In [ ]:
# Pick start on L1 (top-left) and end on L2 (bottom-right)
l1_xy = floor_valid[FLOOR_HEIGHTS[0]]
l2_xy = floor_valid[FLOOR_HEIGHTS[1]]

# L1 start: near min-X, max-Y
i_start = find_closest_node(l1_xy, UMIN + 2, VMAX - 2)
start_key = rk(l1_xy[i_start, 0], l1_xy[i_start, 1])
start_v = all_v[l1_idx[start_key]]

# L2 end: near max-X, min-Y
i_end = find_closest_node(l2_xy, UMAX - 2, VMIN + 2)
end_key = rk(l2_xy[i_end, 0], l2_xy[i_end, 1])
end_v = all_v[l2_idx[end_key]]

print(f'Start (L1): {Vertex.X(start_v):.1f}, {Vertex.Y(start_v):.1f}')
print(f'End   (L2): {Vertex.X(end_v):.1f}, {Vertex.Y(end_v):.1f}')

t0 = time.time()
shortest_path = Graph.ShortestPath(building_graph, vertexA=start_v, vertexB=end_v)
print(f'Shortest path computed in {time.time()-t0:.2f}s')
if shortest_path:
    print(f'  Path length: {Wire.Length(shortest_path):.2f}')
    for edge in Topology.Edges(shortest_path):
        edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width','color'], [5,'red']))
    Topology.Show(building_graph, shortest_path,
                  camera=[1,1,1],
                  vertexSize=1,
                  vertexColor='grey',
                  edgeColor='darkgrey',
                  edgeColorKey='color',
                  edgeWidthKey='width',
                  backgroundColor='black',
                  width=900, height=600,
                  renderer=renderer)
else:
    print('  No path found — check stair connections.')

## 13. Degree Centrality

In [ ]:
degree_values = Graph.DegreeCentrality(building_graph)
heatmap_from_values(degree_values, 'Degree Centrality')

## 14. Closeness Centrality / Integration
Measures how close a node is to all other nodes — corresponds to **global integration** in space syntax.

In [ ]:
closeness_values = Graph.ClosenessCentrality(building_graph, colorScale='thermal')
heatmap_from_values(closeness_values, 'Closeness Centrality')

## 15. Betweenness Centrality / Choice
Measures how often a node lies on shortest paths between other nodes — identifies circulation bottlenecks.

In [ ]:
betweenness_values = Graph.BetweennessCentrality(building_graph, normalize=True, colorScale='thermal')
heatmap_from_values(betweenness_values, 'Betweenness Centrality')

## 16. Clustering Coefficient
Measures how interconnected a node's neighbors are — reveals tightly-knit spatial clusters.

In [ ]:
clustering_values = Graph.ClusteringCoefficient(building_graph)
heatmap_from_values(clustering_values, 'Clustering Coefficient')

## 17. Interpretation

| Metric | What it reveals |
|--------|----------------|
| **Degree Centrality** | Rooms with the most direct connections — high values at corridor/open-plan areas |
| **Closeness Centrality** | Accessibility — which spaces can reach everywhere most efficiently (global integration) |
| **Betweenness Centrality** | Circulation bottlenecks — spaces that control movement flow between others (choice) |
| **Clustering Coefficient** | Spatial hierarchy — tightly clustered private zones vs open connective areas |
| **Shortest Path** | Cross-floor connectivity via stairs — the actual route between floors |

Stair locations should show elevated **betweenness** (they are the only cross-floor connections) and moderate **closeness** (they bridge both floors).